# Similarity measures

## Set up working environment

In [1]:
import xml.etree.ElementTree as ET
import networkx as nx
import math

## Wordnet package

In [2]:
from nltk.corpus import wordnet as wn
import nltk 

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /home/mmathe/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/mmathe/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
# Look up a word using synsets() + optional pos argument to constrin the PoS word
wn.synsets("cat")

[Synset('cat.n.01'),
 Synset('guy.n.01'),
 Synset('cat.n.03'),
 Synset('kat.n.01'),
 Synset('cat-o'-nine-tails.n.01'),
 Synset('caterpillar.n.02'),
 Synset('big_cat.n.01'),
 Synset('computerized_tomography.n.01'),
 Synset('cat.v.01'),
 Synset('vomit.v.01')]

In [4]:
wn.synsets("cat", pos = wn.VERB)

[Synset('cat.v.01'), Synset('vomit.v.01')]

In [5]:
print(wn.synset("cat.n.01").definition())

feline mammal usually having thick soft fur and no ability to roar: domestic cats; wildcats


In [6]:
print(wn.synset("cat.n.01").lemmas())

[Lemma('cat.n.01.cat'), Lemma('cat.n.01.true_cat')]


In [7]:
wn.synset('cat.n.01').lowest_common_hypernyms(wn.synset('bird.n.01'))

[Synset('vertebrate.n.01')]

In [8]:
cat = wn.synset("cat.n.01")
bird = wn.synset("bird.n.01")
cat.path_similarity(bird)

0.14285714285714285

## Load MeSH thesaurus tree

In [9]:
# PARSE MESH THESAURUS TREE
G = nx.DiGraph()
name_to_trees = {}
tree_to_name  = {}

tree = ET.parse("/home/mmathe/Téléchargements/desc2026.xml")
root = tree.getroot()

for record in root.findall("DescriptorRecord"):
    name = record.findtext("DescriptorName/String")
    tree_numbers = [tn.text for tn in record.findall("TreeNumberList/TreeNumber")]

    name_to_trees[name] = tree_numbers
    for tn in tree_numbers:
        tree_to_name[tn] = name

        parts = tn.split(".")
        # Extract letter branch: "A05" -> letter="A", rest="05"
        letter = parts[0][0]        # "A"
        first  = parts[0]           # "A05"

        # ROOT -> A (branch letter)
        G.add_edge("ROOT", letter)
        # A -> A05
        G.add_edge(letter, first)

        # A05 -> A05.810 -> A05.810.453 ...
        for i in range(1, len(parts)):
            parent = ".".join(parts[:i])
            child  = ".".join(parts[:i+1])
            G.add_edge(parent, child)

depths = nx.single_source_shortest_path_length(G, "ROOT")

In [10]:
depths_by_name = {}
for name, trees in name_to_trees.items():
    if not trees:
        depths_by_name[name] = None
    else:
        depths_by_name[name] = max(depths.get(tn, 0) for tn in trees)

In [11]:
print(name_to_trees["Kidney"])
print(name_to_trees["Liver"])

['A05.810.453']
['A03.620']


## Path-based similarity measures

### Rada (1898)

R. Rada, H. Mili, E. Bicknell and M. Blettner, **Development and application of a metric on semantic networks** in IEEE Transactions on Systems, Man, and Cybernetics, vol. 19, no. 1, pp. 17-30, Jan.-Feb. 1989, doi: 10.1109/21.24528.  
Conceptual distance measure = length of the shortest path between 2 concepts in MeSH thesaurus using broader / narrower relations.  

In [12]:
def rada(mesh1, mesh2) :
    trees1 = name_to_trees.get(mesh1)
    trees2 = name_to_trees.get(mesh2)

    if not trees1:
        print(f"'{mesh1}' not found in MeSH")
        return None
    if not trees2:
        print(f"'{mesh2}' not found in MeSH")
        return None

    shortest = float("inf")
    for tn1 in trees1:
        for tn2 in trees2:
            try:
                d = nx.shortest_path_length(G.to_undirected(), tn1, tn2)
                shortest = min(shortest, d)
            except nx.NetworkXNoPath:
                continue

    return shortest if shortest < float("inf") else None

In [13]:
pairs = [
    ("Kidney", "Liver"),
    ("Kidney Diseases", "Liver Diseases"),
    ("Kidney", "Kidney Diseases"),
    ("Hypertension", "Diabetes Mellitus"),
    ("Kidney", "Bacteria"),
]

for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {rada(c1, c2)}")

Kidney                         <-> Liver                          : 5
Kidney Diseases                <-> Liver Diseases                 : 5
Kidney                         <-> Kidney Diseases                : 8
Hypertension                   <-> Diabetes Mellitus              : 5
Kidney                         <-> Bacteria                       : 6


### Caviedes and Cimino (2004)

Jorge E. Caviedes, James J. Cimino, **Towards the development of a conceptual distance metric for the UMLS**. Journal of Biomedical Informatics. Volume 37, Issue 2. 2004. Pages 77-85. ISSN 1532-0464. doi: https://doi.org/10.1016/j.jbi.2004.02.001.  
Same as Rada, but returns $\frac{1}{len(shortestPath(c_1, c_2))}$  
Values in $(0,1]$ : closer terms have a higher score (inverse of distance)

In [14]:
def caviedes_cimino(mesh1, mesh2) :
    if mesh1 == mesh2:
        return 1.0
    
    d = rada(mesh1, mesh2)

    if d is None or d == 0:
        return None
    
    return 1 / d

In [15]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {caviedes_cimino(c1, c2)}")

Kidney                         <-> Liver                          : 0.2
Kidney Diseases                <-> Liver Diseases                 : 0.2
Kidney                         <-> Kidney Diseases                : 0.125
Hypertension                   <-> Diabetes Mellitus              : 0.2
Kidney                         <-> Bacteria                       : 0.16666666666666666


In [16]:
# path similarity in wordnet
def get_best_path_similarity(mesh1, mesh2):
    """
    Compute the best path similarity between two terms
    by trying all synset combinations and taking the maximum.
    """
    synsets1 = wn.synsets(mesh1.replace(" ", "_"))
    synsets2 = wn.synsets(mesh2.replace(" ", "_"))

    if not synsets1:
        print(f"  No synsets found for: '{mesh1}'")
        return None
    if not synsets2:
        print(f"  No synsets found for: '{mesh2}'")
        return None

    best_score = 0
    best_pair = None

    for s1 in synsets1:
        for s2 in synsets2:
            score = s1.path_similarity(s2)
            if score is not None and score > best_score:
                best_score = score
                best_pair = (s1, s2)

    return best_score if best_score > 0 else None

In [17]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_best_path_similarity(c1, c2)}")

Kidney                         <-> Liver                          : 0.25
Kidney Diseases                <-> Liver Diseases                 : 0.14285714285714285
Kidney                         <-> Kidney Diseases                : 0.05263157894736842
Hypertension                   <-> Diabetes Mellitus              : 0.08333333333333333
Kidney                         <-> Bacteria                       : 0.07142857142857142


### Wu and Palmer (1994)

Zhibiao Wu and Martha Palmer. 1994. **Verb Semantics and Lexical Selection.** In 32nd Annual Meeting of the Association for Computational Linguistics, pages 133–138, Las Cruces, New Mexico, USA. Association for Computational Linguistics.  
Least Common Subsummer (LCS) = most specific concept 2 concepts share as an ancestor  
$Sim_{WP} = \frac{2 * depth(LCS(c_1, c_2))}{depth(c_1) + depth(c_2)}$

In [18]:
def mesh_depth(term):
    trees = name_to_trees.get(term)
    if not trees:
        print(f"'{term}' not found in MeSH")
        return None
    return max(depths.get(tn, 0) for tn in trees)

In [19]:
def lowest_common_subsumer(mesh1, mesh2):
    trees1 = name_to_trees.get(mesh1)
    trees2 = name_to_trees.get(mesh2)

    if not trees1:
        print(f"'{mesh1}' not found in MeSH")
        return None
    if not trees2:
        print(f"'{mesh2}' not found in MeSH")
        return None

    best_lcs  = None
    best_depth = -1

    for tn1 in trees1:
        for tn2 in trees2:
            ancestors1 = nx.ancestors(G, tn1) | {tn1}
            ancestors2 = nx.ancestors(G, tn2) | {tn2}
            common = ancestors1 & ancestors2
            if not common:
                continue

            lcs = max(common, key=lambda n: depths.get(n, 0))
            d   = depths.get(lcs, 0)
            if d > best_depth:
                best_depth = d
                best_lcs   = lcs

    # Return the term name if it exists, otherwise the tree number
    name = tree_to_name.get(best_lcs, best_lcs)
    return name, best_lcs

In [20]:
def wu_palmer(mesh1, mesh2) :
    d1 = mesh_depth(mesh1)
    d2 = mesh_depth(mesh2)
    lcs, lcs_tn = lowest_common_subsumer(mesh1, mesh2)
    dlcs = depths.get(lcs_tn)
    if dlcs is None or (d1 + d2) == 0:
        return None
    wp = (2 * dlcs) / (d1 + d2)
    return wp

In [21]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {wu_palmer(c1, c2)}")

Kidney                         <-> Liver                          : 0.2857142857142857
Kidney Diseases                <-> Liver Diseases                 : 0.2222222222222222
Kidney                         <-> Kidney Diseases                : 0.0
Hypertension                   <-> Diabetes Mellitus              : 0.2222222222222222
Kidney                         <-> Bacteria                       : 0.0


In [22]:
# wup similarity in wordnet
def get_wup_similarity(mesh1, mesh2):
    synsets1 = wn.synsets(mesh1.replace(" ", "_"))
    synsets2 = wn.synsets(mesh2.replace(" ", "_"))

    if not synsets1 or not synsets2:
        return None

    best_score = 0
    for s1 in synsets1:
        for s2 in synsets2:
            score = s1.wup_similarity(s2)
            if score is not None and score > best_score:
                best_score = score

    return best_score if best_score > 0 else None

In [23]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_wup_similarity(c1, c2)}")

Kidney                         <-> Liver                          : 0.8235294117647058
Kidney Diseases                <-> Liver Diseases                 : 0.7272727272727273
Kidney                         <-> Kidney Diseases                : 0.1
Hypertension                   <-> Diabetes Mellitus              : 0.5217391304347826
Kidney                         <-> Bacteria                       : 0.23529411764705882


### Leacock and Chodorov (1998)

Same as Wu and Palmer, but adds the depth of the taxonomy $D$  
$Sim_{LC} = - log(\frac{shortestPath(c_1, c_2)}{2D})$

In [24]:
max_depth = max(depths.values())
print(f"Max depth: {max_depth}")

Max depth: 14


In [25]:
def leacock_chodorow(mesh1, mesh2) :
    sp = rada(mesh1, mesh2)
    return -math.log(sp/(2*max_depth))

In [26]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {leacock_chodorow(c1, c2)}")

Kidney                         <-> Liver                          : 1.7227665977411035
Kidney Diseases                <-> Liver Diseases                 : 1.7227665977411035
Kidney                         <-> Kidney Diseases                : 1.252762968495368
Hypertension                   <-> Diabetes Mellitus              : 1.7227665977411035
Kidney                         <-> Bacteria                       : 1.540445040947149


In [27]:
# lch similarity in wordnet
def get_lch_similarity(mesh1, mesh2):
    synsets1 = wn.synsets(mesh1.replace(" ", "_"))
    synsets2 = wn.synsets(mesh2.replace(" ", "_"))

    if not synsets1 or not synsets2:
        return None

    best_score = 0
    for s1 in synsets1:
        for s2 in synsets2:
            if s1.pos() != s2.pos():   # ← skip mismatched POS
                continue
            score = s1.lch_similarity(s2)
            if score is not None and score > best_score:
                best_score = score

    return best_score if best_score > 0 else None

In [28]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_lch_similarity(c1, c2)}")

Kidney                         <-> Liver                          : 2.2512917986064953
Kidney Diseases                <-> Liver Diseases                 : 1.6916760106710724
Kidney                         <-> Kidney Diseases                : 0.6931471805599453
Hypertension                   <-> Diabetes Mellitus              : 1.1526795099383855
Kidney                         <-> Bacteria                       : 0.9985288301111273


### Nguyen and Al Mubaid (2006)

H. Al-Mubaid and H. A. Nguyen, "A Cluster-Based Approach for Semantic Similarity in the Biomedical Domain," 2006 International Conference of the IEEE Engineering in Medicine and Biology Society, New York, NY, USA, 2006, pp. 2713-2717, doi: 10.1109/IEMBS.2006.259235.  
Combine taxonomy depth + LCS  
$Sim_{NAM} = log (2 + (shortestPath(c_1, c_2) - 1) * (D - d))$ with $D$ as the depth of the taxonomy and $d$ as the depth of the LCS

In [29]:
def nguyen_almubaid(mesh1, mesh2) :
    sp = rada(mesh1, mesh2)
    lcs, lcs_tn = lowest_common_subsumer(mesh1, mesh2)
    dlcs = depths.get(lcs_tn)
    if dlcs is None :
        return None
    return math.log((2 + sp) * (max_depth - dlcs))

In [30]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {nguyen_almubaid(c1, c2)}")

Kidney                         <-> Liver                          : 4.51085950651685
Kidney Diseases                <-> Liver Diseases                 : 4.51085950651685
Kidney                         <-> Kidney Diseases                : 4.941642422609304
Hypertension                   <-> Diabetes Mellitus              : 4.51085950651685
Kidney                         <-> Bacteria                       : 4.718498871295094


## Information Content based similarity measures

**Information Content** : can be calculated using information derived from a corpus or information derived from a taxonomy   
  
- Corpus-based :  
IC = negative log of the probability of a concept   
$P(c*) = P(c) + \sum_{descendants(c) P(d)}$  
$P(d) = \frac{freq(d)}{N}$ with $freq(d)$ as the number of times the concept is seen in the corpus and $N$ as the total number of concepts  

- Taxonomy-based : assess the informativeness of a concept based on its placement within the hierarchy by looking at its incoming (ancestors) and outgoing (descendant) links  
$IC(c) = -log(\frac{\frac{|leaves(c)|}{|subsummers(c)|} + 1}{max_leaves + 1})$  

### Resnik (1995) 

Resnik, P. (1995). **Using information content to evaluate semantic similarity in a taxonomy.** arXiv preprint cmp-lg/9511007.  
Modified IC to be used as a similarity measure : similarity of 2 concepts = IC of their LCS  
$Sim_{Res} = IC(LCS(c_1, c_2)) = -log(P(LCS(c_1, c_2)))$  
--> Derive IC from the ontology structure, terms with fewer descendants are more specific and have a higher IC

In [31]:
def get_descendants(tn):
    """All descendant tree numbers of a given tree number."""
    return {n for n in G.nodes if n.startswith(tn + ".") or n == tn}

In [32]:
def compute_ic_structure():
    leaves = {n for n in G.nodes if G.out_degree(n) == 0}
    total_leaves = len(leaves)

    # Propagate leaves up to all ancestors once
    leaf_count = {n: set() for n in G.nodes}
    for leaf in leaves:
        for ancestor in nx.ancestors(G, leaf) | {leaf}:
            leaf_count[ancestor].add(leaf)

    ic = {}

    # IC for named descriptors
    for name, trees in name_to_trees.items():
        if not trees:
            continue
        best_ic = 0
        for tn in trees:
            term_leaves = leaf_count.get(tn, set())
            if not term_leaves:
                continue
            p = len(term_leaves) / total_leaves
            ic_val = -math.log(p) if p > 0 else 0
            best_ic = max(best_ic, ic_val)
        ic[name] = best_ic

    # IC for artificial branch nodes (A, B, C, ...)
    for node in G.nodes:
        if len(node) == 1 and node.isalpha():  # single letter = branch
            term_leaves = leaf_count.get(node, set())
            if term_leaves:
                p = len(term_leaves) / total_leaves
                ic[node] = -math.log(p) if p > 0 else 0

    # IC for ROOT = 0 by definition
    ic["ROOT"] = 0.0

    return ic

ic = compute_ic_structure()

In [33]:
def resnik(mesh1, mesh2):
    """Resnik similarity = IC of the Lowest Common Subsumer."""
    if mesh1 == mesh2 :
        return ic.get(mesh1, 0.0)

    lcs, lcs_tn = lowest_common_subsumer(mesh1, mesh2)

    if lcs_tn == "ROOT":
        return 0.0

    # Branch node (single letter) -> look up by tree number directly
    if len(lcs_tn) == 1:
        return ic.get(lcs_tn, 0.0)

    # Named descriptor -> look up by name
    return ic.get(lcs, 0.0)

In [34]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {resnik(c1, c2)}")

Kidney                         <-> Liver                          : 3.026722902717324
Kidney Diseases                <-> Liver Diseases                 : 1.6153855863577544
Kidney                         <-> Kidney Diseases                : 0.0
Hypertension                   <-> Diabetes Mellitus              : 1.6153855863577544
Kidney                         <-> Bacteria                       : 0.0


In [35]:
# resnik similarity with wordnet
from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic') 
brown_ic = wordnet_ic.ic("ic-brown.dat")

[nltk_data] Downloading package wordnet_ic to
[nltk_data]     /home/mmathe/nltk_data...
[nltk_data]   Package wordnet_ic is already up-to-date!


In [36]:
def get_resnik_similarity(mesh1, mesh2):
    synsets1 = wn.synsets(mesh1.replace(" ", "_"))
    synsets2 = wn.synsets(mesh2.replace(" ", "_"))

    if not synsets1 or not synsets2:
        return None

    best_score = 0
    for s1 in synsets1:
        for s2 in synsets2:
            if s1.pos() != s2.pos():   
                continue
            score = s1.res_similarity(s2, brown_ic)
            if score is not None and score > best_score:
                best_score = score

    return best_score if best_score > 0 else None

In [37]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_resnik_similarity(c1, c2)}")

Kidney                         <-> Liver                          : 8.557516974893456
Kidney Diseases                <-> Liver Diseases                 : 6.715415750804551
Kidney                         <-> Kidney Diseases                : None
Hypertension                   <-> Diabetes Mellitus              : 6.015615471339601
Kidney                         <-> Bacteria                       : 0.8017591149538994


### Lin (1998)

Dekang Lin. 1998. **An Information-Theoretic Definition of Similarity.** In Proceedings of the Fifteenth International Conference on Machine Learning (ICML '98). Morgan Kaufmann Publishers Inc., San Francisco, CA, USA, 296–304.  
Add IC of the individual concepts, similar to Wu and Palmer, but using IC rather than the depth of concepts :  
$Sim_{Lin} = \frac{2 * IC(LCS(c_1, c_2))}{IC(c_1) + IC(c_2)}$

In [38]:
def lin(mesh1, mesh2) :
    if mesh1 == mesh2 :
        return ic.get(mesh1, 0.0)
    
    ic1 = ic.get(mesh1)
    ic2 = ic.get(mesh2)

    lcs, lcs_tn = lowest_common_subsumer(mesh1, mesh2)

    if lcs_tn == "ROOT":
        return 0.0

    # Branch node (single letter) -> look up by tree number directly
    if len(lcs_tn) == 1:
        iclcs = ic.get(lcs_tn, 0.0)
    
    return (2 * iclcs) / (ic1 + ic2)

In [39]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {lin(c1, c2)}")

Kidney                         <-> Liver                          : 0.33506599656721997
Kidney Diseases                <-> Liver Diseases                 : 0.24210349990232524
Kidney                         <-> Kidney Diseases                : 0.0
Hypertension                   <-> Diabetes Mellitus              : 0.18579364510307664
Kidney                         <-> Bacteria                       : 0.0


In [40]:
# lin similarity in wordnet
def get_lin_similarity(mesh1, mesh2):
    synsets1 = wn.synsets(mesh1.replace(" ", "_"))
    synsets2 = wn.synsets(mesh2.replace(" ", "_"))

    if not synsets1 or not synsets2:
        return None

    best_score = 0
    for s1 in synsets1:
        for s2 in synsets2:
            if s1.pos() != s2.pos():   
                continue
            score = s1.lin_similarity(s2, brown_ic)
            if score is not None and score > best_score:
                best_score = score

    return best_score if best_score > 0 else None

In [41]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_lin_similarity(c1, c2)}")

Kidney                         <-> Liver                          : 0.7183568199006088
Kidney Diseases                <-> Liver Diseases                 : 1.3430831501609101e-299
Kidney                         <-> Kidney Diseases                : None
Hypertension                   <-> Diabetes Mellitus              : 1.20312309426792e-299
Kidney                         <-> Bacteria                       : 0.07697062018760453


### Jiang and Conrath (1997)

Jiang, J.J., & Conrath, D.W. (1997). **Semantic Similarity Based on Corpus Statistics and Lexical Taxonomy.** ROCLING/IJCLCLP.  
Modified to return a similarity score by taking the reciprocal of the distance :  
$Sim_{JC} = \frac{1}{IC(c_1) + IC(c_2) - 2 * IC(LCS(c_1, c_2))}$

In [42]:
def jiang_conrath(mesh1, mesh2) :
    if mesh1 == mesh2 :
        return ic.get(mesh1, 0.0)
    
    ic1 = ic.get(mesh1)
    ic2 = ic.get(mesh2)

    lcs, lcs_tn = lowest_common_subsumer(mesh1, mesh2)

    if lcs_tn == "ROOT":
        return 0.0

    # Branch node (single letter) -> look up by tree number directly
    if len(lcs_tn) == 1:
        iclcs = ic.get(lcs_tn, 0.0)
    
    return 1 / ((ic1 + ic2) - (2 * iclcs))

In [43]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {jiang_conrath(c1, c2)}")

Kidney                         <-> Liver                          : 0.08324327449784931
Kidney Diseases                <-> Liver Diseases                 : 0.09887465179780926
Kidney                         <-> Kidney Diseases                : 0.0
Hypertension                   <-> Diabetes Mellitus              : 0.07063015647717187
Kidney                         <-> Bacteria                       : 0.0


In [44]:
# jiang conrath similarity with wordnet

def get_jcn_similarity(mesh1, mesh2):
    synsets1 = wn.synsets(mesh1.replace(" ", "_"))
    synsets2 = wn.synsets(mesh2.replace(" ", "_"))

    if not synsets1 or not synsets2:
        return None

    best_score = 0
    for s1 in synsets1:
        for s2 in synsets2:
            if s1.pos() != s2.pos():   
                continue
            score = s1.jcn_similarity(s2, brown_ic)
            if score is not None and score > best_score:
                best_score = score

    return best_score if best_score > 0 else None

In [45]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_jcn_similarity(c1, c2)}")

Kidney                         <-> Liver                          : 0.14902640152278462
Kidney Diseases                <-> Liver Diseases                 : 1e-300
Kidney                         <-> Kidney Diseases                : 1e-300
Hypertension                   <-> Diabetes Mellitus              : 1e-300
Kidney                         <-> Bacteria                       : 0.05200385749006155


## Relatedness measures

### Lesk (1986)

Michael Lesk. 1986. **Automatic sense disambiguation using machine readable dictionaries: how to tell a pine cone from an ice cream cone.** In Proceedings of the 5th annual international conference on Systems documentation (SIGDOC '86). Association for Computing Machinery, New York, NY, USA, 24–26. https://doi.org/10.1145/318723.318728  
Determine the relatedness between 2 concepts by counting the number of overlaps between their definitions.  
Overlap = longest sequence of consecutive words occuring in both definitions.  
--> Implemented in *Wordnet*

In [46]:
from nltk.wsd import lesk

Lesk algo doesn't return a similarity score directly --> picks the best matching synset for a given word. Adaptation : use each cpncept as context for the other, then get the disambiguated synsets and compute wup similariy

In [47]:
def get_lesk_similarity(mesh1, mesh2):
    # Use each term as context for the other
    synset1 = lesk(mesh1.split(), mesh1.split()[0])
    synset2 = lesk(mesh2.split(), mesh2.split()[0])

    if not synset1:
        print(f"  No synset found for: '{mesh1}'")
        return None
    if not synset2:
        print(f"  No synset found for: '{mesh2}'")
        return None

    print(f"  Synset 1: {synset1.name()} — {synset1.definition()}")
    print(f"  Synset 2: {synset2.name()} — {synset2.definition()}")

    # Once we have the disambiguated synsets, compute wup similarity ( = get a score between 0 and 1)
    score = synset1.wup_similarity(synset2)
    return score

In [48]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_lesk_similarity(c1, c2)}")

  Synset 1: kidney.n.01 — either of two bean-shaped excretory organs that filter wastes (especially urea) from the blood and excrete them and water in urine
  Synset 2: liver.n.01 — large and complicated reddish-brown glandular organ located in the upper right portion of the abdominal cavity; secretes bile and functions in metabolism of protein and carbohydrate and fat; synthesizes substances involved in the clotting of the blood; synthesizes vitamin A; detoxifies poisonous substances and breaks down worn-out erythrocytes
Kidney                         <-> Liver                          : 0.8235294117647058
  Synset 1: kidney.n.01 — either of two bean-shaped excretory organs that filter wastes (especially urea) from the blood and excrete them and water in urine
  Synset 2: liver.n.01 — large and complicated reddish-brown glandular organ located in the upper right portion of the abdominal cavity; secretes bile and functions in metabolism of protein and carbohydrate and fat; synthesizes 

### Liu (2012)

Ying Liu, Bridget T. McInnes, Ted Pedersen, Genevieve Melton-Meaux, and Serguei Pakhomov. 2012. Semantic relatedness study using second order co-occurrence vectors computed from biomedical corpora, UMLS and WordNet. In Proceedings of the 2nd ACM SIGHIT International Health Informatics Symposium (IHI '12). Association for Computing Machinery, New York, NY, USA, 363–372. https://doi.org/10.1145/2110363.2110405  
- Use second-order co-occurrence vectors  
- A vector is created for each word in the concepts definition containing words that co-occur with it in a corpus  
- These vectors are averaged to create a single co-occurrence vector for the concept  
- Similarity between concepts = cosine between second orders vectors

In [49]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

In [50]:
def get_extended_synonyms(mesh):
    synsets = wn.synsets(mesh.replace(" ", "_"))
    if not synsets:
        return ""
    
    syn_words = []
    s = synsets[0]  # use first (most common) sense
    
    # Own gloss
    syn_words.append(s.definition())
    
    # Related synsets: hypernyms, hyponyms, also_sees, etc.
    for related in s.hypernyms() + s.hyponyms() + s.also_sees():
        syn_words.append(related.definition())

    return " ".join(syn_words)

In [51]:
def get_sov_similarity(mesh1, mesh2):
    gloss1 = get_extended_synonyms(mesh1)
    gloss2 = get_extended_synonyms(mesh2)

    if not gloss1 or not gloss2:
        return None

    # Build co-occurrence vectors using TF as a proxy
    vectorizer = CountVectorizer().fit([gloss1, gloss2])
    vectors = vectorizer.transform([gloss1, gloss2]).toarray()

    score = cosine_similarity([vectors[0]], [vectors[1]])[0][0]
    return score

In [52]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_sov_similarity(c1, c2)}")

Kidney                         <-> Liver                          : 0.5143167692487138
Kidney Diseases                <-> Liver Diseases                 : 0.5794402645484684
Kidney                         <-> Kidney Diseases                : 0.34554737023254406
Hypertension                   <-> Diabetes Mellitus              : 0.4769585240283266
Kidney                         <-> Bacteria                       : 0.3924504313164433


/!\ the true Liu et al. method uses a biomedical corpus to build co-occurrence vectors, not just Wordnet glosses 

## FORVM edges as a proxy for similarity

### Check if SKOS:RELATED exists between 2 concepts

In [53]:
from SPARQLWrapper import SPARQLWrapper, JSON

In [54]:
# Define the endpoint URL
forvm_endpoint_url = "https://forum.semantic-metabolomics.fr/sparql/"

In [55]:
def get_forvm_relation(mesh1, mesh2) :   
 # Check if a skos:related relation exists between the 2 concepts
    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    ASK {{
        ?mesh1 rdfs:label "{mesh1}"@en .
        ?mesh1 a meshv:TopicalDescriptor .
        ?mesh2 rdfs:label "{mesh2}"@en .
        ?mesh2 a meshv:TopicalDescriptor .
        ?mesh1 skos:related ?mesh2 .
    }}
    """

    # Initialise the SPARQL wrapper, submit the query, and request JSON back
    sparql = SPARQLWrapper(forvm_endpoint_url)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    result = sparql.query().convert()

    return result["boolean"]

In [56]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_forvm_relation(c1, c2)}")

Kidney                         <-> Liver                          : True
Kidney Diseases                <-> Liver Diseases                 : True
Kidney                         <-> Kidney Diseases                : True
Hypertension                   <-> Diabetes Mellitus              : True
Kidney                         <-> Bacteria                       : False


### Count number of shared descriptors

In [57]:
def get_forvm_shared(mesh1, mesh2) :   
 # Check if a skos:related relation exists between the 2 concepts
    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT COUNT(DISTINCT ?shared) AS ?c
    WHERE {{
        ?mesh1 rdfs:label "{mesh1}"@en .
        ?mesh1 a meshv:TopicalDescriptor .
        ?mesh2 rdfs:label "{mesh2}"@en .
        ?mesh2 a meshv:TopicalDescriptor .
        ?shared skos:related ?mesh1 .
        ?shared skos:related ?mesh2 .
    }}
    """

    # Initialise the SPARQL wrapper, submit the query, and request JSON back
    sparql = SPARQLWrapper(forvm_endpoint_url)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    for result in results["results"]["bindings"] :
        return int(result["c"]["value"])


In [58]:
for c1, c2 in pairs:
    print(f"{c1:30s} <-> {c2:30s} : {get_forvm_shared(c1, c2)}")

Kidney                         <-> Liver                          : 932
Kidney Diseases                <-> Liver Diseases                 : 654
Kidney                         <-> Kidney Diseases                : 1454
Hypertension                   <-> Diabetes Mellitus              : 673
Kidney                         <-> Bacteria                       : 432


In [59]:
import pandas as pd

In [60]:
indexing_results_dataframe = pd.read_csv('indexing_results.tsv',sep = '\t')

In [61]:
indexing_results_dataframe_titles = pd.read_csv("indexing_results_titles.tsv", sep = "\t")

In [62]:
aop_wiki_results = pd.read_csv("ke_mesh_from_aopwiki.csv", sep = ",")
print(len(aop_wiki_results))

142


In [63]:
aop_mesh_mappings = pd.read_csv("aop_mesh_mappings.tsv", sep = "\t")

In [64]:
merged = aop_wiki_results.merge(aop_mesh_mappings, on=['KE_ID', 'MESH_ID'], how='left', indicator=True)
missing = merged[merged['_merge'] == 'left_only'][['KE_ID', 'MESH_ID']]

print(len(missing))

61


## Embeddings

https://github.com/helboukkouri/mesh-embeddings

In [65]:
import gzip
import pickle

with open('mesh_ui_to_id.pickle', 'rb') as stream:
    mesh_ui_to_id = pickle.load(stream)

embeddings = {}
with gzip.open('mesh_embeddings.txt.gz', 'rt') as stream:
    n_embeddings, embedding_dim = stream.readline().strip().split()
    for line in stream:
        splitline = str(line).strip().split()
        idx = int(splitline[0])
        vector = list(map(float, splitline[1:]))
        embeddings[idx] = vector

def get_embedding_from_mesh_ui(mesh_ui):
    return embeddings[mesh_ui_to_id[mesh_ui]]

print(f'There are {n_embeddings} MeSH embeddings')
print(f'Each embedding is {embedding_dim}-dimensional')

print('MeSH UI for <Headache> is: D006261')
print('MeSH embedding for <Headache> is:', get_embedding_from_mesh_ui('D006261'))

There are 29638 MeSH embeddings
Each embedding is 256-dimensional
MeSH UI for <Headache> is: D006261
MeSH embedding for <Headache> is: [-0.22856602, -0.32223737, -0.16364807, 0.16428398, -0.12691289, 0.5625101, -0.17584483, -0.47571012, -0.088811584, 0.2617807, 0.05132794, 0.71627855, 0.43066362, -0.1670893, -0.26608157, -0.09317504, 0.14667332, -0.11840302, -0.1663675, -0.52484214, 0.14188018, -0.24755822, 0.17012611, 0.09081254, -0.12909845, -0.41837966, -0.6537568, 0.371565, 0.12833919, 0.005427597, 0.20416279, 0.08012802, 0.10650572, 0.07799014, 0.044482708, -0.017883368, -0.08914792, -0.16249849, 0.21305005, 0.30060294, -0.14664921, 0.2694519, -0.4759495, -0.42735684, 0.21831755, -0.0701913, -0.54911405, -0.1254092, 0.22662424, -0.06484091, -0.032748558, 0.4300212, -0.12393516, 0.46268645, 0.545198, 0.3075909, -0.44722748, 0.1205208, -0.59417015, 0.31570944, 0.4535999, -0.16720925, 0.1467588, -0.41351464, 0.052442025, 0.30475795, 0.22588854, -0.45776254, -0.21493755, 0.30782074, -